# Fidelidade da porta CPHASE de um sítio de Brod-Combes com molas

Este notebook explica `calcular.py` e `plot.py`. Para cada variável e cada expressão ele diz o que ela representa, mostra a fórmula e indica **qual função do código a implementa**. As células de código importam `calcular.py` e reproduzem o cálculo. Elas **não** sobrescrevem os arquivos `.npz` nem a figura.

**O sistema.** Há um único sítio de interação com dois átomos, $a$ e $b$, ligados por um acoplamento cross-Kerr $\chi$. O fóton $a$ interage com o átomo $a$ e o fóton $b$ com o átomo $b$. Cada átomo está acoplado a uma "mola", um oscilador de frequência $\omega^{(a)}$ ou $\omega^{(b)}$ com acoplamento $g^{(a)}$ ou $g^{(b)}$. As molas começam num estado de Fock $|n_a\rangle$, $|n_b\rangle$.

**Por que não dá para usar a fórmula de Brod & Combes.** No artigo, o estado de saída dos fótons é **puro**, e a fidelidade média se reduz a $F_1=[6+3\,\mathrm{Re}(e^{i\phi}F)+|F|^2]/10$, com um único número complexo $F$. Aqui um fóton pode deixar a mola em $|m\rangle\neq|n\rangle$. Como não medimos a mola, ela é traçada e o estado dos fótons fica **misto**. Usamos então a definição geral de fidelidade entre um estado puro e um misto ([Wikipedia: Fidelity of quantum states](https://en.wikipedia.org/wiki/Fidelity_of_quantum_states)):
$$F(\psi_{\rm id},\rho)=\langle\psi_{\rm id}|\rho|\psi_{\rm id}\rangle .$$
O alvo $|\psi_{\rm id}\rangle$ (porta CPHASE ideal) é puro. Quem é misto é o estado real $\rho$.

| arquivo | o que faz |
|---|---|
| `calcular.py` | calcula $F_1(\sigma)$ e $F_2(\sigma)$ para cada caso de `casos` e salva em `data/*.npz` |
| `plot.py` | lê os `.npz` e desenha a figura no estilo da Fig. 5 do artigo (`fidelidade_sigma_mola.svg/.png`) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import calcular as c

## 1. Parâmetros (topo de `calcular.py`)

Os parâmetros do sítio são os mesmos da Fig. 5 de Brod & Combes ($\chi=5$, $\gamma=4.5$, $\omega_0=1.1$). Assim, com a mola desligada, a curva *single site* do artigo é reproduzida.

| variável no código | símbolo | significado |
|---|---|---|
| `chi` | $\chi$ | acoplamento não linear (cross-Kerr) entre os átomos $a$ e $b$ |
| `gam` | $\gamma$ | taxa de emissão de cada átomo no guia (largura de linha) |
| `Delta` | $\Delta$ | dessintonia dos átomos |
| `omega0` | $\omega_0$ | dessintonia da frequência central do pacote em relação a $\Delta$ |
| `omega_c` | $\omega_c=\Delta+\omega_0$ | frequência central do pacote (derivada) |
| `phi` | $\phi$ | fase da porta CPHASE ideal ($\pi$) |
| `referencia` | — | como se define o fóton "ideal" de saída (seção 4) |
| `sigma_min`, `sigma_max`, `n_sigma` | $\sigma$ | varredura logarítmica da largura do pacote |
| `k_sigma` | — | meia-janela de frequências $= k_\sigma\,\sigma$ |
| `pts_per_width` | — | pontos da grade por menor escala, $\min(\sigma,\gamma)$ |
| `n_min`, `n_max` | — | limites do número de pontos da grade |
| `eps_fc` | — | $|F_{kn}|$ abaixo disso é tratado como zero (corte da soma em $k$ e $m$) |
| `casos` | $(\omega^{(a)},S_a,n_a,\omega^{(b)},S_b,n_b)$ | uma curva por tupla |

A mola é especificada pelo **fator de Huang-Rhys** $S=(2g/\omega)^2$ em vez de $g$. Dele saem $g=\sqrt{S}\,\omega/2$, o deslocamento $\beta=2g/\omega=\sqrt S$ e o deslocamento polarônico da linha, $4g^2/\omega=S\omega$.

In [ ]:
print('chi =', c.chi, ' gamma =', c.gam, ' Delta =', c.Delta, ' omega0 =', c.omega0, ' phi =', c.phi)
print('casos =', c.casos)

## 2. A mola: fatores de Franck-Condon — `FC(k, n, beta)` e `mola(w, S, n)`

$$F_{kn}=\langle k|D(\beta)|n\rangle,\qquad \beta=\frac{2g}{\omega}.$$

Para $k\ge n$:
$$F_{kn}=\sqrt{\frac{n!}{k!}}\;\beta^{k-n}\,e^{-\beta^2/2}\,L_n^{(k-n)}(\beta^2),$$
e $F_{kn}=(-1)^{n-k}F_{nk}$ para $k<n$ (**`FC`**).

**`mola(w, S, n)`** devolve um dicionário com tudo que descreve uma mola:

| chave | significado |
|---|---|
| `w`, `S`, `n`, `g` | $\omega$, $S$, estado de Fock inicial $n$, $g=\sqrt S\,\omega/2$ |
| `desloc` | $4g^2/\omega=S\omega$ (deslocamento polarônico) |
| `F` | matriz $F_{km}$, $0\le k,m<K$ |
| `K` | corte do espaço da mola, aumentado até a faixa útil não encostar nele |
| `j` | valores possíveis de $j=k-n$ (só $k$ com $|F_{kn}|>$ `eps_fc`) |
| `d` | valores possíveis de $d=m-n$ (os $m$ alcançáveis desses $k$) |
| `c` | $c_j=F_{n+j,n}^2$, pesos da amplitude elástica (seção 4) |

O ponto central para a eficiência é que $\Gamma_{kn}$ depende só de $j=k-n$ e o desvio de frequência do fóton só de $d=m-n$. Todas as integrais são calculadas uma vez como funções de $(j,d)$. O $n$ absoluto entra apenas pelos pesos $F$.

In [ ]:
m = c.mola(1.0, 0.1, 1)
print('K =', m['K'], '  j =', m['j'], '\nd =', m['d'])
print('sum_k F_kn^2 =', np.sum(m['F'][:, m['n']]**2))   # deve ser 1 (D(beta) e unitario)

## 3. Amplitude de um fóton — `Gam(ml, j, nu)` e `t_mn(ml, m, n, nu)`

$$\Gamma_{kn}(\nu)=\frac{\gamma}{2}+i\Big(\Delta-\frac{4g^2}{\omega}+(k-n)\,\omega-\nu\Big)\qquad(\texttt{Gam}, \text{ com } j=k-n)$$

$$t_{mn}(\nu)=\delta_{mn}-\gamma\sum_k\frac{F_{km}F_{kn}}{\Gamma_{kn}(\nu)}\qquad(\texttt{t\_mn})$$

O fóton que entra com frequência $\nu$ e leva a mola de $|n\rangle$ a $|m\rangle$ sai com $\omega=\nu-(m-n)\,\omega^{\rm mola}$ (conservação de energia, a $\delta$ do primeiro termo de $S$).

Verificação: a probabilidade se conserva, $\sum_m\int d\nu\,|\xi(\nu)|^2|t_{mn}(\nu)|^2=1$.

In [ ]:
sigma = 1.0
w, h = c.grade(sigma)
xi = c.xi_in(w, sigma)
print('norma de 1 foton =', sum(np.sum(xi**2 * np.abs(c.t_mn(m, mm, m['n'], w))**2) * h for mm in range(m['K'])))

## 4. O pacote e o fóton "ideal" — `xi_in`, `fase_ref`, `grade`

**Pacote de entrada** (Eq. (4) do artigo, **`xi_in`**):
$$\xi(\omega)=\frac{1}{(2\pi\sigma^2)^{1/4}}\exp\!\Big[-\frac{(\omega-\omega_c)^2}{4\sigma^2}\Big].$$

**Remoção da deformação de um fóton.** Brod & Combes comparam a saída com o fóton deformado pelo processo linear, $\xi\to-\frac{\Gamma^*}{\Gamma}\xi$, uma fase pura. Com a mola, a parte elástica $t_{nn}$ não tem mais módulo 1. Por isso o fóton ideal de saída é definido como
$$\tilde\xi(\omega)=u(\omega)\,\xi(\omega),\qquad |u|=1,$$
com duas opções (variável `referencia`, função **`fase_ref`**):

* `'media'` (padrão): $u=t_{nn}/|t_{nn}|$, a fase da amplitude elástica, calculada por $t_{nn}(\nu)=1-\gamma\sum_j c_j/\Gamma_j(\nu)$. Para $g=0$ dá exatamente $-\Gamma^*/\Gamma$, como no artigo.
* `'sem_mola'`: $u=-\Gamma_0^*/\Gamma_0$, com $\Gamma_0=\gamma/2+i(\Delta-\nu)$, a fase do artigo sem mola.

**Grade** (**`grade`**): janela $\omega_c\pm k_\sigma\sigma$ e espaçamento $h\approx\min(\sigma,\gamma)/$`pts_per_width`. As gaussianas confinam o integrando à janela, e o espaçamento resolve tanto o pacote quanto as linhas de largura $\gamma$.

In [ ]:
nu = np.linspace(-15, 15, 600)
plt.figure(figsize=(6, 3))
plt.plot(nu, np.angle(c.fase_ref(m, nu)), label='fase de $t_{nn}$ (mola)')
plt.plot(nu, np.angle(c.fase_ref(c.mola(1, 0, 0), nu)), '--', label=r'$-\Gamma^*/\Gamma$ (sem mola)')
plt.xlabel(r'$\nu$'); plt.ylabel('arg u'); plt.legend(); plt.tight_layout(); plt.show()

: 

## 5. Amplitudes lineares projetadas — `amplitudes_1foton`

A amplitude de o fóton $a$ (sozinho) sair igual ao fóton ideal **e** deixar a mola em $|m\rangle$ é
$$\tau_{mn}=\int d\nu\;\tilde\xi^*\big(\nu-(m-n)\omega\big)\,\xi(\nu)\,t_{mn}(\nu)
=\delta_{mn}\,O-\gamma\sum_k F_{km}F_{kn}\,L[k-n,\,m-n],$$
$$O=\int d\nu\,\tilde\xi^*(\nu)\xi(\nu),\qquad L[j,d]=\int d\nu\,\frac{\tilde\xi^*(\nu-d\,\omega)\,\xi(\nu)}{\Gamma_j(\nu)}.$$

`amplitudes_1foton` devolve o vetor `tau[m]` $=\tau_{m n}$ para o $n$ inicial da mola. Para $m\ne n$ o fóton sai deslocado de $(m-n)\omega$. Só há sobreposição com $\tilde\xi$ se esse deslocamento for comparável a $\sigma$.

In [ ]:
tau = c.amplitudes_1foton(m, sigma, w, h)
for mm in range(5):
    print(f'm={mm}:  tau = {tau[mm]:.4f}   |tau|^2 = {abs(tau[mm])**2:.4f}')

## 6. Termo não linear — `tabela_T` e `termo_nao_linear`

Projetando o segundo termo de $S$ sobre $\tilde\xi_a\tilde\xi_b$ e integrando sobre $\xi(\nu_a)\xi(\nu_b)$:
$$N_{m_am_b}=-\frac{i\chi\gamma^2}{\pi}\sum_{k_a,k_b}F^{(a)}_{k_am_a}F^{(a)}_{k_an_a}F^{(b)}_{k_bm_b}F^{(b)}_{k_bn_b}\;
T[k_a-n_a,\,k_b-n_b,\,m_a-n_a,\,m_b-n_b].$$

**Troca de variáveis.** Com $\omega_a'=\omega_a+d_a\omega^{(a)}$ e $\omega_b'=\omega_b+d_b\omega^{(b)}$ ($d=m-n$):
* a $\delta$ vira $\omega_a'+\omega_b'=\nu_a+\nu_b\equiv E$;
* $\Gamma_{k m}(\omega)=\Gamma_{k n}(\omega')$, de modo que todos os denominadores dependem só de $j=k-n$.

Logo
$$T[j_a,j_b,d_a,d_b]=\int dE\;K_f(E)\,P(E)\,Q(E),$$
$$P(E)=\int d\nu\,\frac{\xi(\nu)}{\Gamma^{(a)}_{j_a}(\nu)}\frac{\xi(E-\nu)}{\Gamma^{(b)}_{j_b}(E-\nu)},\qquad
Q(E)=\int d\omega'\,\frac{\tilde\xi^*(\omega'-d_a\omega^{(a)})}{\Gamma^{(a)}_{j_a}(\omega')}\frac{\tilde\xi^*(E-\omega'-d_b\omega^{(b)})}{\Gamma^{(b)}_{j_b}(E-\omega')},$$
$$K_f(E)=\Big(1+\frac{2i\chi}{\Sigma}\Big)^{-1},\qquad \Sigma_{k_ak_b}=\Gamma^{(a)}_{k_an_a}(\nu_a)+\Gamma^{(b)}_{k_bn_b}(\nu_b)=\gamma+i\big(2\Delta-S_a\omega^{(a)}-S_b\omega^{(b)}+j_a\omega^{(a)}+j_b\omega^{(b)}-E\big),$$
que depende só de $E$.

**Implementação (`tabela_T`).**
* $P$ e $Q$ são convoluções 1D calculadas por FFT. A integral tripla original vira convoluções, o mesmo truque de `fidelidade_sigma/calcular.py`.
* As funções de saída são amostradas na grade de entrada transladada de $i_d=\mathrm{round}(d\,\omega/h)$ pontos (função **`saida`**), para que $Q$ caia na mesma grade de $E$ que $P$. O valor de $\tilde\xi^*$ é calculado no ponto exato, então isso não é uma aproximação.
* $\int dE\,K_fPQ$ é feito por Parseval no espaço de Fourier, com comprimento $N_f\ge4n$ (sem aliasing).
* $T$ é calculado **uma vez por $\sigma$**. Depois, `termo_nao_linear` só soma com os pesos $F_{km}F_{kn}$.

**Validação.** Comparado com a integral tripla direta (Simpson 3D) para molas diferentes em $a$ e $b$ ($\omega=0.7, 1.3$; $S=0.3, 0.2$), o erro relativo ficou em $\sim10^{-8}$.

In [ ]:
mb = c.mola(1.0, 0.1, 1)
T = c.tabela_T(m, mb, sigma, w, h)
N = c.termo_nao_linear(m, mb, T)
print('T.shape =', T.shape, '   N[n_a, n_b] =', N[m['n'], mb['n']])

## 7. Fidelidade média de porta (estado misto) — `fidelidades`, `W1`, `W2`

Entrada lógica: $|\psi\rangle=\sum_k c_k|k\rangle$, com $k\in\{00,01,10,11\}$ (ausência/presença do fóton $a$ e $b$). As molas começam em $|n_an_b\rangle$. O estado ideal é
$$|\psi_{\rm id}\rangle=c_{00}|00\rangle+c_{01}|0\tilde\xi\rangle+c_{10}|\tilde\xi0\rangle+e^{i\phi}c_{11}|\tilde\xi\tilde\xi\rangle .$$

O estado real dos fótons, com a mola traçada, é $\rho=\sum_{m_am_b}|{\rm out}_{m}\rangle\langle{\rm out}_{m}|$, em que $|{\rm out}_m\rangle=\langle m_am_b|U|\psi,n_an_b\rangle$. Então
$$F=\langle\psi_{\rm id}|\rho|\psi_{\rm id}\rangle=\sum_{m_a,m_b}\Big|\sum_k|c_k|^2x_k(m)\Big|^2,$$
$$x_{00}=\delta_{m_an_a}\delta_{m_bn_b},\quad x_{01}=\delta_{m_an_a}\tau^{(b)}_{m_bn_b},\quad x_{10}=\tau^{(a)}_{m_an_a}\delta_{m_bn_b},\quad x_{11}=e^{i\phi}\big(\tau^{(a)}_{m_an_a}\tau^{(b)}_{m_bn_b}+N_{m_am_b}\big).$$

Os ramos com $m\neq n$ não interferem com o ramo $|00\rangle$, em que a mola não muda. É assim que a mola, ao "registrar" a passagem do fóton, destrói a coerência e torna o estado misto.

**Média sobre os estados de entrada.** $\mathbb E\big|\sum_k|c_k|^2x_k\big|^2=x^\dagger Wx$:
* Haar em 2 qubits ($d=4$): $\mathbb E|c_k|^4=\tfrac1{10}$, $\mathbb E|c_j|^2|c_k|^2=\tfrac1{20}$, logo $W_1=(J+I)/20$ (**`W1`**), que dá $F_1$;
* Haar em estados produto: $\mathbb E[p_ip_k]=(1+\delta_{ik})/6$ por qubit, logo $W_2=\tfrac1{36}M\otimes M$ com $M=\begin{pmatrix}2&1\\1&2\end{pmatrix}$ (**`W2`**), que dá $F_2$.

$$F_{1,2}=\sum_{m_a,m_b}x(m)^\dagger\,W_{1,2}\,x(m).$$

**Limite sem mola.** Com $m=n$ apenas, $\tau=1$ e $x=(1,1,1,e^{i\phi}F)$, o que recupera exatamente as Eqs. (A2) e (A5) do artigo:
$F_1=\frac{6+3\mathrm{Re}(e^{i\phi}F)+|F|^2}{10}$ e $F_2=\frac{11+5\mathrm{Re}(e^{i\phi}F)+2|F|^2}{18}$.

In [ ]:
for Fv in [-1, 1, 0.3 + 0.2j]:
    x = np.array([1, 1, 1, np.exp(1j * c.phi) * Fv])
    print(f'F={Fv}:  x^T W1 x = {np.real(x.conj() @ c.W1 @ x):.6f}   (A2) = {(6 + 3*np.real(np.exp(1j*c.phi)*Fv) + abs(Fv)**2)/10:.6f}'
          f'   |  x^T W2 x = {np.real(x.conj() @ c.W2 @ x):.6f}   (A5) = {(11 + 5*np.real(np.exp(1j*c.phi)*Fv) + 2*abs(Fv)**2)/18:.6f}')

m0 = c.mola(1.0, 0.0, 0)          # S = 0: sem mola
print('sem mola, sigma=2:', c.fidelidades(m0, m0, 2.0), '  (pico ~0.78 da curva single site da Fig. 5)')
print('mola S=0.1, n=1, sigma=2:', c.fidelidades(m, mb, 2.0))

## 8. Varredura e saída — `calcular_caso`, `nome_arquivo`, `main`

`calcular_caso(w_a, S_a, n_a, w_b, S_b, n_b)` percorre `n_sigma` valores de $\sigma$ em escala log, entre `sigma_min` e `sigma_max`, e calcula `fidelidades`. Ele avisa se $F_1>1$ (sinal de grade insuficiente) e salva em `data/` um `.npz` com `sigmas`, `F1`, `F2` e todos os parâmetros. O nome do arquivo vem de `nome_arquivo`, montado a partir dos parâmetros. `main()` roda todos os `casos`.

Para gerar os dados, rode `python3 calcular.py` no terminal. Leva alguns segundos por caso. A célula abaixo faz uma versão curta, sem salvar nada.

In [ ]:
sig = np.logspace(np.log10(0.03), 2, 25)
plt.figure(figsize=(6, 3.5))
for (S, n, cor) in [(0.0, 0, 'red'), (0.1, 0, 'black'), (0.1, 2, '#1E90FF')]:
    ma_ = c.mola(1.0, S, n)
    F12 = np.array([c.fidelidades(ma_, ma_, s) for s in sig])
    plt.semilogx(sig, F12[:, 0], '-', color=cor, label=f'$F_1$  S={S}, n={n}')
    plt.semilogx(sig, F12[:, 1], '--', color=cor, label=f'$F_2$  S={S}, n={n}')
plt.xlabel(r'$\sigma$'); plt.ylabel('Fidelidade'); plt.legend(fontsize=7); plt.tight_layout(); plt.show()

## 9. A figura — `plot.py`

`plot.py` lê os `.npz` listados em `curvas` (arquivo, cor, rótulo) e desenha, como na Fig. 5 do artigo, $F_1$ em **linha cheia** e $F_2$ em **tracejado**, com $\sigma$ em escala log. Ele salva `fidelidade_sigma_mola.svg` e `.png`. Para acrescentar uma curva: adicione a tupla em `casos` (`calcular.py`), rode `calcular.py` e acrescente o arquivo em `curvas`.

**Leitura dos resultados** ($\chi=5$, $\gamma=4.5$, $\omega_0=1.1$, $\omega^{(a,b)}=1$, $S=0.1$):
* A curva sem mola reproduz a curva *single site* da Fig. 5 (pico $F_1\approx0.78$ em $\sigma\approx2$).
* A mola reduz a fidelidade, e o efeito cresce com o estado de Fock inicial $n$. O peso elástico $F_{nn}^2=e^{-\beta^2}[L_n(\beta^2)]^2$ diminui com $n$, e as transições $n\to m\ne n$ ficam mais prováveis.
* Para $\sigma\gg\gamma$ todas as curvas tendem a $0.4$ ($F_1$) e $0.444$ ($F_2$), como no artigo: o pacote largo não acumula a fase não linear.
* Para $\sigma\ll\omega$ a mola pesa mais, porque as bandas laterais ($m\ne n$) saem espectralmente distinguíveis do fóton ideal e contam como erro.

In [ ]:
from IPython.display import Image
Image('fidelidade_sigma_mola.png')